# Day 3 — Model Training & In-Distribution Evaluation

Trains models on the **temporal SCM v1 aggregated data** and evaluates them on the **normal half** of the test set.  
*(Note: XGBoost has been replaced with sklearn GradientBoostingClassifier to ensure maximum stability and avoid DLL kernel crashes).* 

| Model | Features | Strategy |
|---|---|---|
| **GBM-All** | All 22 features (including spurious) | Boosted trees |
| **Causal-LR (observable)** | 6 causal observable features | Logistic Regression |
| **Causal-LR (behavioural)** | 8 features (+ agency, consistency) | Logistic Regression |

In [ ]:
# ── Cell 1: Imports & Setup ───────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings('ignore')

# Ensure output directories exist relative to the working directory
os.makedirs('results', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('Setup complete. No XGBoost dependencies used to prevent kernel crashes.')

Setup complete. No XGBoost dependencies used to prevent kernel crashes.


In [1]:
import os
import pandas as pd

# 1. Plug in (mount) your Google Drive to the Colab virtual machine
from google.colab import drive
drive.mount('/content/drive')

# 2. Update the paths to include the Colab-specific Drive path
paths_to_try = [
    '/content/drive/MyDrive/credit_score', # <-- Google Drive path in Colab
    '/content/drive/MyDrive/Credit_Score', # (Added just in case the folder has capital letters)
    '.',   
    'data', 
]

train_path, test_path = None, None
for base in paths_to_try:
    candidate_train = os.path.join(base, 'temporal_credit_agg_train.csv')
    candidate_test  = os.path.join(base, 'temporal_credit_agg_test.csv')
    if os.path.exists(candidate_train) and os.path.exists(candidate_test):
        train_path = candidate_train
        test_path = candidate_test
        print(f"Found datasets inside: {base}")
        break

if train_path is None:
    raise FileNotFoundError("Dataset files not found in any of the expected directories!")

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print(f"\nTrain shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(f"\nTest split counts:\n{test['split'].value_counts()}")
print(f"\nColumns ({len(train.columns)}): {list(train.columns)}")

Mounted at /content/drive
Found datasets inside: /content/drive/MyDrive/credit_score

Train shape: (10000, 26)
Test  shape: (10000, 26)

Test split counts:
split
normal       5000
recession    5000
Name: count, dtype: int64

Columns (26): ['age_bucket', 'employment_status', 'household_structure', 'financial_agency', 'financial_consistency', 'digital_footprint_mean', 'income_mean', 'income_cv', 'income_trend', 'utility_rate', 'utility_recent', 'dti_final', 'dti_mean', 'sc_final', 'sc_trend', 'shock_total', 'shock_recent', 'peer_shock_exposure', 'default', 'dark_mode_user', 'signup_weekend', 'social_media_score', 'geolocation_cluster', 'app_diversity_index', 'num_inquiries', 'split']


In [ ]:
# ── Cell 3: Split Normal vs Recession ─────────────────────────────────────────
test_normal    = test[test['split'] == 'normal'].copy()
test_recession = test[test['split'] == 'recession'].copy()

LABEL = 'default'

X_train      = train.drop(columns=[LABEL, 'split'], errors='ignore')
y_train      = train[LABEL]
X_test_norm  = test_normal.drop(columns=[LABEL, 'split'], errors='ignore')
y_test_norm  = test_normal[LABEL]
X_test_rec   = test_recession.drop(columns=[LABEL, 'split'], errors='ignore')
y_test_rec   = test_recession[LABEL]

print(f"X_train:     {X_train.shape}")
print(f"X_test_norm: {X_test_norm.shape}")
print(f"X_test_rec:  {X_test_rec.shape}")
print(f"\nDefault rates  | train: {y_train.mean():.2%} | normal: {y_test_norm.mean():.2%} | recession: {y_test_rec.mean():.2%}")

: 

In [ ]:
# ── Cell 4: Feature Sets ──────────────────────────────────────────────────────
# Mirrors TEMPORAL_FEATURE_SETS from scm_temporal_v1.py
FEATURE_SETS = {
    'xgboost_all': [
        'age_bucket', 'employment_status', 'household_structure',
        'income_mean', 'income_cv', 'income_trend',
        'utility_rate', 'utility_recent',
        'dti_final', 'dti_mean',
        'shock_total', 'shock_recent',
        'sc_final', 'sc_trend',
        'peer_shock_exposure',
        'digital_footprint_mean',
        # Spurious zone — correlations REVERSE in recession
        'dark_mode_user', 'signup_weekend', 'social_media_score',
        'geolocation_cluster', 'app_diversity_index', 'num_inquiries',
    ],
    'causal_lr_observable': [
        'income_mean', 'income_cv',
        'utility_rate',
        'dti_final',
        'employment_status',
        'shock_total',
    ],
    'causal_lr_behavioural': [
        'income_mean', 'income_cv',
        'utility_rate',
        'dti_final',
        'employment_status',
        'shock_total',
        'financial_agency',
        'financial_consistency',
    ],
}

SPURIOUS_COLS = [
    'dark_mode_user', 'signup_weekend', 'social_media_score',
    'geolocation_cluster', 'app_diversity_index', 'num_inquiries'
]

for name, feats in FEATURE_SETS.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# ── Cell 5: Validation Checks ─────────────────────────────────────────────────
sep = '=' * 58

print(sep)
print('CHECK 1 — Data integrity')
print(sep)
assert train.shape[0] == 10_000
assert test.shape[0]  == 10_000
assert test_normal.shape[0]    == 5_000
assert test_recession.shape[0] == 5_000
assert LABEL in train.columns and LABEL in test.columns
print('  [OK] Shape checks passed.')

print()
print(sep)
print('CHECK 2 — Feature set completeness')
print(sep)
for name, features in FEATURE_SETS.items():
    missing = set(features) - set(train.columns)
    assert len(missing) == 0, f'Missing columns in {name}: {missing}'
    print(f'  [OK] {name}: all {len(features)} features present')

print()
print(sep)
print('CHECK 3 — No data leakage (feature mean diff < 0.1)')
print(sep)
bad_cols = []
for col in train.columns:
    if col not in ['default', 'split']:
        diff = abs(train[col].mean() - test_normal[col].mean())
        if diff >= 0.10:
            bad_cols.append((col, diff))
if bad_cols:
    print(f'  [WARN] {len(bad_cols)} columns exceed 0.1 threshold:')
    for c, d in bad_cols:
        print(f'    {c}: diff={d:.4f}')
else:
    print('  [OK] All feature means within tolerance.')

print()
print('All checks passed!')

In [ ]:
# ── Cell 6: Train GBM-All (GradientBoostingClassifier) ────────────────────────
feats_xgb = FEATURE_SETS['xgboost_all']

print('Training GradientBoostingClassifier-All...')
booster = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42)

booster.fit(X_train[feats_xgb], y_train)

prob_xgb_norm  = booster.predict_proba(X_test_norm[feats_xgb])[:, 1]
prob_xgb_train = booster.predict_proba(X_train[feats_xgb])[:, 1]
auc_xgb_norm   = roc_auc_score(y_test_norm, prob_xgb_norm)
auc_xgb_train  = roc_auc_score(y_train, prob_xgb_train)

model_label = 'GBM-All'
print(f'  Train AUC:       {auc_xgb_train:.4f}')
print(f'  Normal Test AUC: {auc_xgb_norm:.4f}')
status = 'OK' if auc_xgb_norm >= 0.80 else ('LOW — check data!' if auc_xgb_norm < 0.70 else 'below expected range')
print(f'  Status: [{status}]')

In [ ]:
# ── Cell 7: Train Causal-LR (observable) ─────────────────────────────────────
print('Training Causal-LR (observable)...')
feats_obs = FEATURE_SETS['causal_lr_observable']

lr_obs = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42))
])
lr_obs.fit(X_train[feats_obs], y_train)

prob_lr_obs_norm  = lr_obs.predict_proba(X_test_norm[feats_obs])[:, 1]
prob_lr_obs_train = lr_obs.predict_proba(X_train[feats_obs])[:, 1]
auc_lr_obs_norm   = roc_auc_score(y_test_norm, prob_lr_obs_norm)
auc_lr_obs_train  = roc_auc_score(y_train, prob_lr_obs_train)

print(f'  Train AUC:       {auc_lr_obs_train:.4f}')
print(f'  Normal Test AUC: {auc_lr_obs_norm:.4f}')
status = 'OK' if auc_lr_obs_norm >= 0.74 else 'LOW — check features!'
print(f'  Status: [{status}]')

In [ ]:
# ── Cell 8: Train Causal-LR (behavioural) ────────────────────────────────────
print('Training Causal-LR (behavioural)...')
feats_beh = FEATURE_SETS['causal_lr_behavioural']

lr_beh = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42))
])
lr_beh.fit(X_train[feats_beh], y_train)

prob_lr_beh_norm  = lr_beh.predict_proba(X_test_norm[feats_beh])[:, 1]
prob_lr_beh_train = lr_beh.predict_proba(X_train[feats_beh])[:, 1]
auc_lr_beh_norm   = roc_auc_score(y_test_norm, prob_lr_beh_norm)
auc_lr_beh_train  = roc_auc_score(y_train, prob_lr_beh_train)

print(f'  Train AUC:       {auc_lr_beh_train:.4f}')
print(f'  Normal Test AUC: {auc_lr_beh_norm:.4f}')
status = 'OK' if auc_lr_beh_norm >= 0.77 else 'LOW — check financial_agency signal!'
print(f'  Status: [{status}]')

In [ ]:
# ── Cell 9: Collect & Save Results ───────────────────────────────────────────
results = pd.DataFrame({
    'model':     ['GBM-All', 'Causal-LR (observable)', 'Causal-LR (behavioural)'],
    'auc_train': [auc_xgb_train, auc_lr_obs_train, auc_lr_beh_train],
    'auc_normal':[auc_xgb_norm,  auc_lr_obs_norm,  auc_lr_beh_norm],
})
results.to_csv('results/baseline_auc.csv', index=False)

# Print Day 3 success table
expected = {
    'GBM-All':                 (0.82, 0.85),
    'Causal-LR (observable)':  (0.76, 0.79),
    'Causal-LR (behavioural)': (0.79, 0.82),
}

header = f"\n{'Model':<30} {'Expected':^12} {'Actual':^8} Status"
print(header)
print('-' * 65)
for _, row in results.iterrows():
    lo, hi = expected[row['model']]
    flag = '✓ OK' if lo <= row['auc_normal'] <= hi else ('▲ High' if row['auc_normal'] > hi else '✗ Low')
    print(f"{row['model']:<30} {lo:.2f}–{hi:.2f}     {row['auc_normal']:.4f}   {flag}")

print(f"\nSaved: results/baseline_auc.csv")

In [ ]:
# ── Cell 10: ROC Curve Plot ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

models_info = [
    (model_label,              prob_xgb_norm,    auc_xgb_norm,    '#e74c3c'),
    ('Causal-LR (observable)', prob_lr_obs_norm, auc_lr_obs_norm, '#2ecc71'),
    ('Causal-LR (beh.)',       prob_lr_beh_norm, auc_lr_beh_norm, '#3498db'),
]
for name, probs, auc, color in models_info:
    fpr, tpr, _ = roc_curve(y_test_norm, probs)
    ax.plot(fpr, tpr, label=f'{name}  (AUC={auc:.4f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Normal Test Set (Day 3 Baseline)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig('outputs/roc_curves_normal.png', dpi=150)
print('Saved: outputs/roc_curves_normal.png')
plt.show()

In [ ]:
# ── Cell 11: Feature Importance Plot ─────────────────────────────────────────
importances = pd.Series(booster.feature_importances_, index=feats_xgb).sort_values()
colors = ['#e74c3c' if f in SPURIOUS_COLS else '#3498db' for f in importances.index]

fig, ax = plt.subplots(figsize=(8, 8))
importances.plot(kind='barh', ax=ax, color=colors)

causal_patch  = mpatches.Patch(color='#3498db', label='Causal feature')
spurious_patch = mpatches.Patch(color='#e74c3c', label='Spurious feature')
ax.legend(handles=[causal_patch, spurious_patch], fontsize=10)
ax.set_title(f'{model_label} Feature Importances', fontsize=13)
ax.set_xlabel('Importance (F-score)', fontsize=11)
plt.tight_layout()
fig.savefig('outputs/feature_importance_xgb.png', dpi=150)
print('Saved: outputs/feature_importance_xgb.png')
plt.show()

In [ ]:
# ── Cell 12: Spurious Reversal Check (Check 5) ───────────────────────────────
print('Spurious Correlation Reversal Check')
print('=' * 65)
hdr = f"{'Variable':<28} {'Train':>8} {'Normal':>8} {'Recession':>10} {'Reversed':>10}"
print('  ' + hdr)
print('  ' + '-' * 65)

log_lines = ['Spurious Correlation Reversal Check', '=' * 65, '  ' + hdr]
all_reversed = True

for col in SPURIOUS_COLS:
    ct = train[col].corr(train['default'].astype(float))
    cn = test_normal[col].corr(test_normal['default'].astype(float))
    cr = test_recession[col].corr(test_recession['default'].astype(float))
    rev = ct * cr < 0
    if not rev:
        all_reversed = False
    line = f"  {col:<28} {ct:>+8.3f} {cn:>+8.3f} {cr:>+10.3f} {'YES' if rev else 'NO ':>10}"
    print(line)
    log_lines.append(line)

summary = f"\n  All 6 reversed: {'YES ✓' if all_reversed else 'NO ✗'}"
print(summary)
log_lines.append(summary)

with open('outputs/spurious_reversal_check.txt', 'w') as f:
    f.write('\n'.join(log_lines))
print('Saved: outputs/spurious_reversal_check.txt')

In [ ]:
# ── Cell 13: Day 3 Summary ────────────────────────────────────────────────────
print('=' * 58)
print('  DAY 3 COMPLETE — Normal Baseline Results')
print('=' * 58)
print(f"  {'Model':<30} {'AUC (normal)':>12}")
print('  ' + '-' * 45)
for _, row in results.iterrows():
    print(f"  {row['model']:<30} {row['auc_normal']:>12.4f}")
print()
print('  Files saved:')
for f in ['results/baseline_auc.csv','outputs/roc_curves_normal.png',
          'outputs/feature_importance_xgb.png','outputs/spurious_reversal_check.txt']:
    print(f'    {f}')
print()
print('  Next: Day 4-5 — evaluate on RECESSION test set & compute stability drop.')